# TAURUS: Anti-TNF scRNA-seq Treatment Validation (Fig 6 / Supp Fig 10)

scRNA-seq analysis of the TAURUS CD anti-TNF treatment cohort (NatureImmunology dataset).
CD cell subset → AUCell annotation → Treg pseudotime (DPT) → Early/Late Treg bins →
patient-level pseudotime delta Pre→Post anti-TNF → fibroblast pre/post comparison →
remission vs non-remission stratification.

**Run `03_taurus_anno_cd.R` first** to generate AUCell annotation and Treg label transfer outputs.


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import scipy.sparse as sp
import scanpy.external as sce
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from matplotlib.lines import Line2D
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from scipy.stats import wilcoxon
import statsmodels.api as sm
# ── 0. Settings ────────────────────────────────────────────────────────────────
sc.settings.verbosity = 2
sc.settings.n_jobs = 8
RANDOM_STATE = 42
 

In [ ]:
adata = sc.read_h5ad('/path/to/taurus/TAURUS_raw_counts_annotated_final.h5ad')

In [ ]:
pd.set_option('display.max_columns', 39)

In [ ]:
adata.obs['Disease'].value_counts()

In [ ]:
cd = adata[adata.obs['Disease']=='CD']

In [ ]:
cd.layers["counts"] = cd.X.copy()

In [ ]:
cd.obs['Remission_status'].value_counts()

In [ ]:
cd.obs['Patient'].value_counts()

In [ ]:
cd.var["mt"] = cd.var_names.str.startswith("MT-")

In [ ]:
sc.pp.calculate_qc_metrics(
    cd,
    qc_vars=["mt",],
    percent_top=None,
    log1p=False,
    inplace=True,
)
 

In [ ]:
sc.pl.violin(cd, ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
             jitter=0.4, multi_panel=True, save="_qc_before.pdf")

In [ ]:
MIN_GENES   = 200     # remove near-empty droplets
MAX_GENES   = 7000    # remove likely doublets (raise for highly complex cells)
MAX_MT_PCT  = 30      # remove dying/lysed cells
MIN_COUNTS  = 500
MAX_COUNTS  = 40000   # dataset-dependent; check the tail
 


In [ ]:
print(f"Cells before filtering: {cd.n_obs}")
 
sc.pp.filter_cells(cd, min_genes=MIN_GENES)
sc.pp.filter_cells(cd, max_genes=MAX_GENES)
sc.pp.filter_cells(cd, min_counts=MIN_COUNTS)
sc.pp.filter_cells(cd, max_counts=MAX_COUNTS)
 
cd = cd[cd.obs["pct_counts_mt"] < MAX_MT_PCT].copy()
 
print(f"Cells after filtering:  {cd.n_obs}")

In [ ]:
raw_counts = cd.layers["counts"]

In [ ]:
cd.obs['Remission_status'].value_counts()

In [ ]:
sc.pp.filter_genes(cd, min_cells=3)

In [ ]:
sc.pp.normalize_total(cd, target_sum=1e4)
sc.pp.log1p(cd)
 

In [ ]:
cd.write_h5ad("/path/to/taurus/xnorm_counts_for_aucell.h5ad.gz", compression="gzip")

In [ ]:
cd=sc.read_h5ad("/path/to/taurus/xnorm_counts_for_aucell.h5ad.gz")

In [ ]:
anno=pd.read_csv('/path/to/taurus/anno/norm_aucell.csv')

In [ ]:
mapper= pd.read_csv('/path/to/scrna/cd/cell_type_category_map.csv')

In [ ]:
# Step 1: Clean the names
mapper['cell_type_aucell'] = mapper['cell type'].str.replace(r'[+\-\/()]', ' ', regex=True)
mapper['cell_type_aucell'] = mapper['cell_type_aucell'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Step 2: Disambiguate duplicates by appending " 1", " 2", etc.
mapper['cell_type_aucell'] = (
    mapper.groupby('cell_type_aucell').cumcount().astype(str).replace('0', '', regex=False)
    .radd(mapper['cell_type_aucell'] + ' ').str.strip()
)

In [ ]:
anno['label_clean'] = anno['label'].str.rstrip()

In [ ]:
anno_map=anno.merge(mapper, how = 'left', left_on = 'label_clean',right_on = 'cell_type_aucell')

In [ ]:
cd.obs['aucell_cell_type_short'] = anno_map['cell type short'].tolist()
cd.obs['aucell_cell_type_full'] = anno_map['cell type'].tolist()
cd.obs['aucell_cell_category_from_type'] = anno_map['cell category'].tolist()
cd.obs['aucell_cell_type_general'] = anno_map['cell type general'].tolist()

In [ ]:
cd.write_h5ad("/path/to/taurus/raw_norm_counts_anno.h5ad.gz", compression="gzip")

In [ ]:
treg=cd[cd.obs['aucell_cell_type_short']=='Treg']

In [ ]:
sc.pp.highly_variable_genes(
    treg, n_top_genes=3000, flavor='seurat_v3', layer='counts'
)

In [ ]:
print(treg.obs["Patient"].value_counts())

In [ ]:
sc.pp.scale(treg, max_value=10)


In [ ]:
sc.tl.pca(treg, n_comps=50, use_highly_variable=True, )
# Harmony integrates on PCA embeddings; writes to X_pca_harmony


In [ ]:
sce.pp.harmony_integrate(treg, key='Patient')  # or 'batch'

sc.pp.neighbors(treg, use_rep='X_pca_harmony', n_neighbors=20, n_pcs=30)
sc.tl.umap(treg)


In [ ]:
panels = {
 'Activated_tissue': ['BATF','PRDM1','MAF','TNFRSF18','TNFRSF4'],
 'Tfr': ['BCL6','CXCR5','PDCD1','ICOS'],
 'Th17_like': ['RORC','CCR6','IL17A','IL17F'],
 'IFN_response': ['ISG15','IFI6','MX1','IFIT3'],
 'Cell_cycle': ['MKI67','TOP2A','STMN1'],
 'Treg_core':['FOXP3','IL2RA','CTLA4','TIGIT','ENTPD1','LRRC32','IL10']
}
for name, genes in panels.items():
    sc.tl.score_genes(treg, [g for g in genes if g in treg.var_names], score_name=name)

In [ ]:
for name in panels.keys():
    sc.pl.umap(
        treg,
        color=name,
        cmap="viridis",     # or 'Reds', 'coolwarm', etc.
        vmin='p1', vmax='p99',  # clip extreme values for nicer contrast
        frameon=False,
    )


In [ ]:
treg_counts =pd.DataFrame(treg.layers['counts'].toarray())
treg_counts.columns = treg.var_names.tolist()
treg_counts.index = treg.obs_names.tolist()

In [ ]:
treg_counts.to_csv('/path/to/taurus/all_raw_treg_counts.csv')


# ap1_targets = ['CCL5', 'CCL4', 'GNLY', 'IL7R', 'CXCR4', 
               'CD69', 'GSTP1', 'BTG1', 'FOS', 'CD8A']

sc.tl.score_genes(treg_pre, ap1_targets, score_name='ap1_score')

# new bins all 

In [ ]:
sc.tl.leiden(treg, resolution=0.4, key_added='leiden_treg_full', random_state=42)

print(f"Clusters: {treg.obs['leiden_treg_full'].nunique()}")

sc.tl.score_genes(treg, ['FOXP3','CTLA4','BATF','IL2RA','CD27'],
                  score_name='suppressive_score')
sc.tl.score_genes(treg, ['NKG7','CCL3','CCL4','CD8A','GZMB'],
                  score_name='effector_score')
sc.tl.score_genes(treg, ['FOS','JUN','JUNB','FOSL1','FOSL2'],
                  score_name='ap1_score')
sc.tl.score_genes(treg, ap1_targets,
                  score_name='ap1_target_score')

sc.pl.umap(treg,
           color=['leiden_treg_full', 'suppressive_score', 'effector_score',
                  'ap1_score', 'ap1_target_score', 'Th17_like', 
                  'Activated_tissue', 'Treg_core', 'IFN_response'],
           ncols=3, frameon=False, cmap='RdBu_r',
           vmin='p1', vmax='p99',
           save='_treg_full_scores.pdf')

In [ ]:
root_cluster = (treg.obs.groupby('leiden_treg_full')['suppressive_score']
                .mean().idxmax())

In [ ]:
root_cell = treg.obs[treg.obs['leiden_treg_full'] == root_cluster].index[0]
treg.uns['iroot'] = treg.obs_names.get_loc(root_cell)

sc.tl.diffmap(treg)
sc.tl.dpt(treg)

# Bin pseudotime
treg.obs['pt_bin_full'] = pd.qcut(treg.obs['dpt_pseudotime'], q=5, labels=False)

In [ ]:
# Patient-level mean pseudotime by timepoint
pt_change = (
    treg.obs.groupby(['Patient', 'Remission_status', 'Treatment'])['dpt_pseudotime']
    .mean()
    .reset_index()
)

# Pivot to get pre and post columns
pt_pivot = pt_change.pivot_table(
    index=['Patient', 'Remission_status'],
    columns='Treatment',
    values='dpt_pseudotime'
).reset_index()

pt_pivot['delta'] = pt_pivot['Post'] - pt_pivot['Pre']
pt_pivot = pt_pivot.dropna()

print(pt_pivot)
print(pt_pivot.groupby('Remission_status')['delta'].describe().round(3))

g1 = pt_pivot[pt_pivot['Remission_status'] == 'Remission']['delta']
g2 = pt_pivot[pt_pivot['Remission_status'] == 'Non_Remission']['delta']
_, p = mannwhitneyu(g2, g1, alternative='two-sided')
print(f"\nRemission delta: {g1.mean():.3f}")
print(f"Non-remission delta: {g2.mean():.3f}")
print(f"p = {p:.3f}")

In [ ]:
sc.pl.umap(treg,
           color=['leiden_treg_full', 'dpt_pseudotime', 'suppressive_score',
                  'ap1_score', 'ap1_target_score',
                  'Treatment', 'Remission_status','Patient'],
           ncols=3, frameon=False, cmap='RdBu_r',
           vmin='p1', vmax='p99',
           save='_treg_full_dpt.pdf')

In [ ]:
# Normalize DPT to 0-1 scale
dpt_min = treg.obs['dpt_pseudotime'].min()
dpt_max = treg.obs['dpt_pseudotime'].max()

treg.obs['dpt_pseudotime_norm'] = (
    (treg.obs['dpt_pseudotime'] - dpt_min) / (dpt_max - dpt_min)
)

print(treg.obs['dpt_pseudotime_norm'].describe().round(3))

# Recompute patient-level stats with normalized score
pt_stats = (
    treg.obs.groupby(['Patient', 'Remission_status', 'Treatment'], observed=True)
    .agg(
        mean_dpt         = ('dpt_pseudotime_norm', 'mean'),
        Disease_duration = ('Disease_duration', 'first'),
    )
    .reset_index()
    .dropna()
)

pt_pivot = pt_stats.pivot_table(
    index=['Patient', 'Remission_status'],
    columns='Treatment',
    values='mean_dpt'
).reset_index().dropna()

pt_pivot['delta'] = pt_pivot['Post'] - pt_pivot['Pre']

# Recheck stats
g1 = pt_pivot[pt_pivot['Remission_status'] == 'Remission']['Pre']
g2 = pt_pivot[pt_pivot['Remission_status'] == 'Non_Remission']['Pre']
_, p_base = mannwhitneyu(g2, g1, alternative='two-sided')

g1d = pt_pivot[pt_pivot['Remission_status'] == 'Remission']['delta']
g2d = pt_pivot[pt_pivot['Remission_status'] == 'Non_Remission']['delta']
_, p_delta = mannwhitneyu(g2d, g1d, alternative='two-sided')

print(f"Baseline rem: {g1.mean():.3f}, nonrem: {g2.mean():.3f}, p={p_base:.3f}")
print(f"Delta rem: {g1d.mean():.3f}, nonrem: {g2d.mean():.3f}, p={p_delta:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
palette = {'Remission': '#378ADD', 'Non_Remission': '#D85A30'}
x_positions = {
    'Remission':     {'Pre': 0,   'Post': 0.7},
    'Non_Remission': {'Pre': 1.2, 'Post': 1.9}
}
for status, color in palette.items():
    subset = pt_pivot[pt_pivot['Remission_status'] == status]
    x_pre  = x_positions[status]['Pre']
    x_post = x_positions[status]['Post']
    for _, row in subset.iterrows():
        ax.plot([x_pre, x_post], [row['Pre'], row['Post']],
                color=color, alpha=0.4, lw=1.2)
        ax.scatter([x_pre, x_post], [row['Pre'], row['Post']],
                   color=color, s=40, zorder=3,
                   edgecolor='white', linewidth=0.8)
    ax.plot([x_pre, x_post],
            [subset['Pre'].mean(), subset['Post'].mean()],
            color=color, lw=3, zorder=4)
    ax.scatter([x_pre, x_post],
               [subset['Pre'].mean(), subset['Post'].mean()],
               color=color, s=100, zorder=5,
               edgecolor='white', linewidth=1.5)

y_max = pt_pivot[['Pre','Post']].max().max()
y_min = pt_pivot[['Pre','Post']].min().min()
y_range = y_max - y_min

# ── Delta p-value (top) ───────────────────────────────────────────────────────
_, p_delta = mannwhitneyu(
    pt_pivot[pt_pivot['Remission_status'] == 'Non_Remission']['delta'],
    pt_pivot[pt_pivot['Remission_status'] == 'Remission']['delta'],
    alternative='two-sided'
)
ax.plot([0.35, 0.35, 1.55, 1.55],
        [y_max+y_range*0.03, y_max+y_range*0.06,
         y_max+y_range*0.06, y_max+y_range*0.03], 'k-', lw=1)
ax.text(0.95, y_max+y_range*0.08, f'Δ p = {p_delta:.3f}',
        ha='center', fontsize=8, color='black')

# ── Pre vs Pre p-value — bracket between x=0 and x=1.2 ──────────────────────
g1_pre = pt_pivot[pt_pivot['Remission_status'] == 'Remission']['Pre']
g2_pre = pt_pivot[pt_pivot['Remission_status'] == 'Non_Remission']['Pre']
_, p_pre = mannwhitneyu(g2_pre, g1_pre, alternative='two-sided')

mid_y = y_min + y_range * 0.05

# Bracket — vertical ticks + horizontal line
mid_y = y_min - y_range * 0.08

bracket_y   = mid_y
tick_height = y_range * 0.025
ax.plot([0, 0],     [bracket_y, bracket_y + tick_height], 'k-', lw=1)
ax.plot([1.2, 1.2], [bracket_y, bracket_y + tick_height], 'k-', lw=1)
ax.plot([0, 1.2],   [bracket_y, bracket_y],               'k-', lw=1)
ax.text(0.6, bracket_y - tick_height*2.2 + y_range*0.001,
        f'Pre p = {p_pre:.3f}',
        ha='center', fontsize=8, color='black')
# ── X axis labels ─────────────────────────────────────────────────────────────
ax.set_xticks([0, 0.7, 1.2, 1.9])
ax.set_xticklabels(['Pre', 'Post', 'Pre', 'Post'])

# Group labels below x-axis
ax.text(0.35, y_min - y_range*0.32, 'Remission',
        ha='center', color='#378ADD', fontsize=9,
        fontweight='bold', transform=ax.transData)
ax.text(1.55, y_min - y_range*0.32, 'Non-remission',
        ha='center', color='#D85A30', fontsize=9,
        fontweight='bold', transform=ax.transData)

ax.set_ylabel('Treg Pseudotime')
ax.set_title('')
ax.set_ylim(y_min - y_range*0.35, y_max + y_range*0.15)
ax.set_xlim(-0.3, 2.2)
sns.despine()
plt.tight_layout()
plt.savefig('treg_pseudotime_lineplot_final.pdf', dpi=300, bbox_inches='tight')
plt.show()

print(f"Pre vs Pre: p = {p_pre:.3f}")
print(f"Delta: p = {p_delta:.3f}")

In [ ]:

# Scores on full treg object
sc.tl.score_genes(treg, ap1_targets, score_name='ap1_target_score')

# Patient-level aggregation
pt_scores = (
    treg.obs.groupby(['Patient', 'Remission_status', 'Treatment'], observed=True)
    .agg(ap1_target=('ap1_target_score', 'median'))
    .reset_index()
    .dropna()
)

fig, ax = plt.subplots(figsize=(5, 3.2))
palette = {'Remission': '#378ADD', 'Non_Remission': '#D85A30'}
x_positions = {
    'Remission':     {'Pre': 0,   'Post': 0.7},
    'Non_Remission': {'Pre': 1.2, 'Post': 1.9}
}

pt_piv = pt_scores.pivot_table(
    index=['Patient', 'Remission_status'],
    columns='Treatment',
    values='ap1_target'
).reset_index().dropna()

for status, color in palette.items():
    subset = pt_piv[pt_piv['Remission_status'] == status]
    x_pre  = x_positions[status]['Pre']
    x_post = x_positions[status]['Post']
    for _, row in subset.iterrows():
        ax.plot([x_pre, x_post], [row['Pre'], row['Post']],
                color=color, alpha=0.4, lw=1.2)
        ax.scatter([x_pre, x_post], [row['Pre'], row['Post']],
                   color=color, s=40, zorder=3,
                   edgecolor='white', linewidth=0.8)
    ax.plot([x_pre, x_post],
            [subset['Pre'].mean(), subset['Post'].mean()],
            color=color, lw=3, zorder=4)
    ax.scatter([x_pre, x_post],
               [subset['Pre'].mean(), subset['Post'].mean()],
               color=color, s=100, zorder=5,
               edgecolor='white', linewidth=1.5)

# Within-group paired tests
rem = pt_piv[pt_piv['Remission_status'] == 'Remission']
non_rem = pt_piv[pt_piv['Remission_status'] == 'Non_Remission']
_, p_rem = wilcoxon(rem['Pre'], rem['Post'])
_, p_nonrem = wilcoxon(non_rem['Pre'], non_rem['Post'])

y_max = pt_piv[['Pre', 'Post']].max().max()
y_min = pt_piv[['Pre', 'Post']].min().min()
y_range = y_max - y_min

# Within-group bracket for Remission
ax.plot([0, 0, 0.7, 0.7],
        [y_max + y_range*0.03, y_max + y_range*0.06,
         y_max + y_range*0.06, y_max + y_range*0.03], 'k-', lw=1)
ax.text(0.35, y_max + y_range*0.08, f'p = {p_rem:.3f}',
        ha='center', fontsize=8)

# Within-group bracket for Non-Remission
ax.plot([1.2, 1.2, 1.9, 1.9],
        [y_max + y_range*0.03, y_max + y_range*0.06,
         y_max + y_range*0.06, y_max + y_range*0.03], 'k-', lw=1)
ax.text(1.55, y_max + y_range*0.08, f'p = {p_nonrem:.3f}',
        ha='center', fontsize=8)

ax.set_xticks([0, 0.7, 1.2, 1.9])
ax.set_xticklabels(['Pre', 'Post', 'Pre', 'Post'])
ax.set_ylabel('AP-1 Target Score')
ax.set_title('')
ax.set_ylim(y_min - y_range*0.20, y_max + y_range*0.15)
ax.set_xlim(-0.3, 2.2)

y_bottom = y_min - y_range*0.17
ax.text(0.35, y_bottom, 'Remission',
        ha='center', color='#378ADD', fontsize=9, fontweight='bold')
ax.text(1.55, y_bottom, 'Non-remission',
        ha='center', color='#D85A30', fontsize=9, fontweight='bold')

sns.despine(ax=ax)
print(f"Remission pre vs post: p={p_rem:.3f}")
print(f"Non-remission pre vs post: p={p_nonrem:.3f}")
plt.tight_layout()
plt.savefig('ap1_target_prepost.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
late_treg_mask = treg.obs['dpt_pseudotime_norm'] > treg.obs['dpt_pseudotime_norm'].quantile(0.75)
treg_late = treg[late_treg_mask].copy()

In [ ]:
# Merge pseudotime delta and AP-1 delta per patient
pt_merged = pt_pivot[['Patient','Remission_status','delta']].merge(
    pt_piv[['Patient','Remission_status','delta']].rename(columns={'delta':'ap1_delta'}),
    on=['Patient','Remission_status']
)

r, p = spearmanr(pt_merged['delta'], pt_merged['ap1_delta'])
print(f"Pseudotime delta vs AP-1 delta: r={r:.3f}, p={p:.3f}")

In [ ]:
sc.tl.score_genes(treg, ['FOXP3','CTLA4','BATF','IL2RA','CD27'],
                  score_name='suppressive_score')

In [ ]:
# Filter weird genes
weird_genes = (
    treg_pre_new.var_names.str.startswith('MT-') |
    treg_pre_new.var_names.str.startswith('RPL') |
    treg_pre_new.var_names.str.startswith('RPS') |
    treg_pre_new.var_names.str.startswith('MTRNR') |
    treg_pre_new.var_names.str.startswith('LINC') |
    treg_pre_new.var_names.str.startswith('AC0') |
    treg_pre_new.var_names.str.startswith('AL') |
    treg_pre_new.var_names.str.startswith('AP0') |
    treg_pre_new.var_names.str.startswith('MIR')
)

treg_clean = treg_pre_new[:, ~weird_genes].copy()

# Make sure pt_bin_new is string categorical
treg_clean.obs['pt_bin_new'] = treg_clean.obs['pt_bin_new'].astype(str).astype('category')

sc.tl.rank_genes_groups(
    treg_clean,
    groupby='pt_bin_new',
    method='wilcoxon',
    use_raw=False,
    key_added='rank_genes_bins'
)

result = sc.get.rank_genes_groups_df(
    treg_clean,
    group=None,
    key='rank_genes_bins'
)

# Top 5 per bin, no repeats
seen_genes = set()
genes_per_bin = {}
for group in ['0', '1', '2', '3', '4']:
    group_genes = []
    for gene in result[result['group'] == group]['names']:
        if gene not in seen_genes:
            group_genes.append(gene)
            seen_genes.add(gene)
        if len(group_genes) == 5:
            break
    genes_per_bin[f'Bin {group}'] = group_genes

print(genes_per_bin)

sc.pl.dotplot(
    treg_clean,
    var_names=genes_per_bin,
    groupby='pt_bin_new',
    standard_scale='var',
    dendrogram=False,
    save='_treg_bins_markers_clean.pdf'
)

In [ ]:
fibro_pre = cd[(cd.obs['Treatment'] == 'Pre') & 
                  (cd.obs['aucell_cell_type_short'].str.contains('Fib', na=False))].copy()

treg_pre_full = cd[(cd.obs['Treatment'] == 'Pre') & 
                      (cd.obs['aucell_cell_type_short'] == 'Treg')].copy()


In [ ]:
# Get raw expression values for single genes
def get_gene_expr(adata_sub, gene, groupby='Patient'):
    if gene not in adata_sub.var_names:
        print(f"{gene} not in var_names")
        return None
    
    gene_idx = adata_sub.var_names.get_loc(gene)
    expr = adata_sub.X[:, gene_idx]
    if hasattr(expr, 'toarray'):
        expr = expr.toarray().flatten()
    
    adata_sub.obs[f'{gene}_expr'] = expr
    return adata_sub.obs.groupby(['Patient', 'Remission_status'])[f'{gene}_expr'].mean().reset_index()

# CD99 in fibroblasts
cd99_pt = get_gene_expr(fibro_pre, 'CD99')
print(cd99_pt)

# CD81 in Tregs
cd81_pt = get_gene_expr(treg_pre_full, 'CD81')
print(cd81_pt)

# Merge and correlate
merged = cd99_pt.merge(cd81_pt, on=['Patient', 'Remission_status'])
merged = merged.dropna()

r, p = spearmanr(merged['CD99_expr'], merged['CD81_expr'])
print(f"\nCD99 (fibroblast) vs CD81 (Treg): r={r:.3f}, p={p:.3f}")
print(merged.groupby('Remission_status')[['CD99_expr','CD81_expr']].mean().round(3))

In [ ]:
# Drop NaN rows — each patient appears twice (one real, one NaN)
merged_clean = merged.dropna()
print(f"n patients: {len(merged_clean)}")


In [ ]:
fibro_all = cd[cd.obs['aucell_cell_type_short'].str.contains('Fib', na=False)].copy()
treg_all  = cd[cd.obs['aucell_cell_type_short'] == 'Treg'].copy()

cd99_pt = get_gene_expr_tx(fibro_all, 'CD99')
cd81_pt = get_gene_expr_tx(treg_all,  'CD81')

merged_tx = cd99_pt.merge(cd81_pt, on=['Patient', 'Remission_status', 'Treatment'])
print(merged_tx.groupby(['Remission_status', 'Treatment'])[['CD99_expr','CD81_expr']].mean().round(3))

In [ ]:
def get_gene_expr_tx(adata_sub, gene):
    if gene not in adata_sub.var_names:
        print(f"{gene} not found")
        return None
    gene_idx = adata_sub.var_names.get_loc(gene)
    expr = adata_sub.X[:, gene_idx]
    if hasattr(expr, 'toarray'):
        expr = expr.toarray().flatten()
    adata_sub.obs[f'{gene}_expr'] = expr
    return (
        adata_sub.obs.groupby(['Patient', 'Remission_status', 'Treatment'],
                               observed=True)[f'{gene}_expr']
        .mean().reset_index().dropna()
    )

cd99_pt = get_gene_expr_tx(fibro_all, 'CD99')
cd81_pt = get_gene_expr_tx(treg_all,  'CD81')

merged_tx = cd99_pt.merge(cd81_pt, on=['Patient', 'Remission_status', 'Treatment'])
print(merged_tx.groupby(['Remission_status', 'Treatment'])[['CD99_expr','CD81_expr']].mean().round(3))

In [ ]:
# Copy dpt_pseudotime_norm from treg to treg_all
treg_all.obs['dpt_pseudotime_norm'] = treg.obs.loc[
    treg_all.obs_names, 'dpt_pseudotime_norm'
]

In [ ]:
merged_pre = merged_sample[merged_sample['Treatment'] == 'Pre'].copy()

print(f"n samples: {len(merged_pre)}")
print(f"n patients: {merged_pre['Patient'].nunique()}")

fig, ax = plt.subplots(figsize=(5, 4))

palette = {'Remission': '#378ADD', 'Non_Remission': '#D85A30'}

for status, color in palette.items():
    mask = merged_pre['Remission_status'] == status
    ax.scatter(
        merged_pre.loc[mask, 'CD99_expr'],
        merged_pre.loc[mask, 'CD81_expr'],
        c=color, s=50, label=status,
        edgecolor='white', linewidth=0.8, alpha=0.8, zorder=3
    )

r, p = spearmanr(merged_pre['CD99_expr'], merged_pre['CD81_expr'])
m, b = np.polyfit(merged_pre['CD99_expr'], merged_pre['CD81_expr'], 1)
x_line = np.linspace(merged_pre['CD99_expr'].min(),
                     merged_pre['CD99_expr'].max(), 100)
ax.plot(x_line, m*x_line + b, 'k--', lw=1.2, alpha=0.6)

g1 = merged_pre[merged_pre['Remission_status'] == 'Remission']['CD81_expr']
g2 = merged_pre[merged_pre['Remission_status'] == 'Non_Remission']['CD81_expr']
_, p_rem = mannwhitneyu(g2, g1, alternative='two-sided')

ax.set_xlabel('CD99 expression (fibroblasts)')
ax.set_ylabel('CD81 expression (Tregs)')
ax.set_title(f'CD99-CD81 co-expression at baseline\n'
             f'Spearman r = {r:.3f}, p = {p:.3f}\n'
             f'(n={len(merged_pre)} samples, {merged_pre["Patient"].nunique()} patients)')
ax.legend(frameon=False, fontsize=9, title='Remission status')
sns.despine()
plt.tight_layout()
plt.savefig('cd99_cd81_baseline.pdf', dpi=300, bbox_inches='tight')
plt.show()

print(f"r = {r:.3f}, p = {p:.3f}")
print(f"Remission vs non-remission CD81: p = {p_rem:.3f}")
print(merged_pre.groupby('Remission_status')[['CD99_expr','CD81_expr']].mean().round(3))

In [ ]:
# ── Setup — get CD99 and CD81 per sample (patient x site x inflammation) ──────
fibro_all = cd[
    cd.obs['aucell_cell_type_short'].str.contains('Fib|Myofib', na=False)
].copy()
treg_all  = cd[cd.obs['aucell_cell_type_short'] == 'Treg'].copy()
treg_all.obs['dpt_pseudotime_norm']=treg.obs['dpt_pseudotime_norm'].tolist()
def get_gene_expr_sample(adata_sub, gene):
    if gene not in adata_sub.var_names:
        print(f"{gene} not found")
        return None
    gene_idx = adata_sub.var_names.get_loc(gene)
    expr = adata_sub.X[:, gene_idx]
    if hasattr(expr, 'toarray'):
        expr = expr.toarray().flatten()
    adata_sub.obs[f'{gene}_expr'] = expr
    return (
        adata_sub.obs.groupby(
            ['Patient', 'Remission_status', 'Treatment', 'Site', 'Inflammation'],
            observed=True
        )[f'{gene}_expr']
        .mean().reset_index().dropna()
    )

cd99_sample = get_gene_expr_sample(fibro_all, 'CD99')
cd81_sample = get_gene_expr_sample(treg_all,  'CD81')

merged_sample = cd99_sample.merge(
    cd81_sample,
    on=['Patient', 'Remission_status', 'Treatment', 'Site', 'Inflammation']
).dropna()

# Add Treg pseudotime per sample
treg_dpt = (
    treg_all.obs.groupby(
        ['Patient', 'Remission_status', 'Treatment', 'Site', 'Inflammation'],
        observed=True
    )['dpt_pseudotime_norm']
    .median().reset_index()
)

merged_sample = merged_sample.merge(
    treg_dpt,
    on=['Patient', 'Remission_status', 'Treatment', 'Site', 'Inflammation']
).dropna()

print(f"n samples: {len(merged_sample)}")
print(merged_sample.groupby('Inflammation').size())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
palette_inflam = {'Inflamed': '#D85A30', 'Non_Inflamed': '#378ADD'}

# ── 2. CD99-CD81 by inflammation status ───────────────────────────────────────
for status, color in palette_inflam.items():
    mask = merged_sample['Inflammation'] == status
    axes[0].scatter(
        merged_sample.loc[mask, 'CD99_expr'],
        merged_sample.loc[mask, 'CD81_expr'],
        c=color, s=50, label=status,
        edgecolor='white', linewidth=0.8, alpha=0.8, zorder=3
    )

# Overall correlation
r, p = spearmanr(merged_sample['CD99_expr'], merged_sample['CD81_expr'])
m, b = np.polyfit(merged_sample['CD99_expr'], merged_sample['CD81_expr'], 1)
x_line = np.linspace(merged_sample['CD99_expr'].min(),
                     merged_sample['CD99_expr'].max(), 100)
axes[0].plot(x_line, m*x_line + b, 'k--', lw=1.2, alpha=0.6)

# Stats by inflammation
inf    = merged_sample[merged_sample['Inflammation'] == 'Inflamed']['CD81_expr']
noninf = merged_sample[merged_sample['Inflammation'] == 'Non_Inflamed']['CD81_expr']
_, p_inf = mannwhitneyu(inf, noninf, alternative='two-sided')

axes[0].set_xlabel('CD99 expression (fibroblasts)')
axes[0].set_ylabel('CD81 expression (Tregs)')
axes[0].set_title(f'CD99-CD81 co-expression\nr = {r:.3f}, p = {p:.3f} | Inflamed vs Non-inflamed p = {p_inf:.3f}')
axes[0].legend(frameon=False, fontsize=9)
sns.despine(ax=axes[0])

# ── 4. CD99-CD81 vs Treg pseudotime ──────────────────────────────────────────
# Create combined LR score
merged_sample['lr_score'] = merged_sample['CD99_expr'] * merged_sample['CD81_expr']

for status, color in palette_inflam.items():
    mask = merged_sample['Inflammation'] == status
    axes[1].scatter(
        merged_sample.loc[mask, 'lr_score'],
        merged_sample.loc[mask, 'dpt_pseudotime_norm'],
        c=color, s=50, label=status,
        edgecolor='white', linewidth=0.8, alpha=0.8, zorder=3
    )

r2, p2 = spearmanr(merged_sample['lr_score'], merged_sample['dpt_pseudotime_norm'])
m2, b2 = np.polyfit(merged_sample['lr_score'], merged_sample['dpt_pseudotime_norm'], 1)
x_line2 = np.linspace(merged_sample['lr_score'].min(),
                      merged_sample['lr_score'].max(), 100)
axes[1].plot(x_line2, m2*x_line2 + b2, 'k--', lw=1.2, alpha=0.6)

axes[1].set_xlabel('CD99 × CD81 LR score')
axes[1].set_ylabel('Treg pseudotime (median)')
axes[1].set_title(f'CD99-CD81 LR score vs Treg pseudotime\nSpearman r = {r2:.3f}, p = {p2:.3f}')
axes[1].legend(frameon=False, fontsize=9)
sns.despine(ax=axes[1])

plt.tight_layout()
plt.savefig('cd99_cd81_inflam_pseudotime.pdf', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nCD99-CD81 correlation: r={r:.3f}, p={p:.3f}")
print(f"CD81 inflamed vs non-inflamed: p={p_inf:.3f}")
print(f"LR score vs pseudotime: r={r2:.3f}, p={p2:.3f}")
print(f"\nCD81 mean by inflammation:")
print(merged_sample.groupby('Inflammation')['CD81_expr'].mean().round(3))

In [ ]:

# 2. Recompute cd99_sample
cd99_sample = get_gene_expr_sample(fibro_all, 'CD99')

# 3. Rebuild merged_sample
merged_sample = cd99_sample.merge(
    cd81_sample,
    on=['Patient', 'Remission_status', 'Treatment', 'Site', 'Inflammation']
).dropna()

# 4. Rebuild merged_pre
merged_pre = merged_sample[merged_sample['Treatment'] == 'Pre'].copy()

# 5. Rebuild pt_piv and pt_combined
pt_piv = merged_sample.pivot_table(
    index=['Patient', 'Remission_status'],
    columns='Treatment',
    values='CD81_expr',
    aggfunc='mean'
).reset_index().dropna()
pt_piv['delta'] = pt_piv['Post'] - pt_piv['Pre']

pt_baseline = merged_pre.groupby(['Patient', 'Remission_status'])[
    ['CD99_expr', 'CD81_expr']
].mean().reset_index()

pt_delta = pt_piv[['Patient', 'Remission_status', 'delta']].copy()
pt_delta.columns = ['Patient', 'Remission_status', 'cd81_delta']

pt_combined = pt_baseline.merge(pt_delta, on=['Patient', 'Remission_status']).dropna()
pt_combined['dot_size'] = (-pt_combined['cd81_delta'] * 500).clip(lower=30)

# 6. Rebuild merged_plot
merged_plot = merged_pre.merge(
    pt_combined[['Patient', 'cd81_delta', 'dot_size']],
    on='Patient'
)

print(f"n patients: {merged_plot['Patient'].nunique()}")
print(sorted(merged_plot['Patient'].unique()))

In [ ]:
# Merge sample-level baseline with patient-level delta
merged_plot = merged_pre.merge(
    pt_combined[['Patient', 'cd81_delta', 'dot_size']],
    on='Patient'
)
fig, ax = plt.subplots(figsize=(5.5, 4.4))
for status, color in palette.items():
    mask = merged_plot['Remission_status'] == status
    ax.scatter(
        merged_plot.loc[mask, 'CD99_expr'],
        merged_plot.loc[mask, 'CD81_expr'],
        c=color,
        s=merged_plot.loc[mask, 'dot_size'].clip(lower=30),  # min size 30
        label=status,
        edgecolor='white', linewidth=0.8,
        alpha=0.7, zorder=3
    )
r, p = spearmanr(merged_plot['CD99_expr'], merged_plot['CD81_expr'])
m, b = np.polyfit(merged_plot['CD99_expr'], merged_plot['CD81_expr'], 1)
x_line = np.linspace(merged_plot['CD99_expr'].min(),
                     merged_plot['CD99_expr'].max(), 100)
ax.plot(x_line, m*x_line + b, 'k--', lw=1.2, alpha=0.6)
# Size legend — use actual values from data
q25 = pt_combined['cd81_delta'].quantile(0.25)
q50 = pt_combined['cd81_delta'].quantile(0.50)
q75 = pt_combined['cd81_delta'].quantile(0.75)
size_legend = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray',
           markersize=np.sqrt(max(-v*500, 30)), 
           label=f'CD81 Δ = {v:.2f}')
    for v in [q25, q50, q75]
]
color_legend = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=color,
           markersize=8, label=status)
    for status, color in palette.items()
]
ax.legend(handles=color_legend + size_legend,
          frameon=False, fontsize=7,
          title='Color = remission\nSize = CD81 Δ pre→post',
          loc='lower right')
ax.set_xlabel('CD99 Expression (Fibroblasts)')
ax.set_ylabel('CD81 Expression (Tregs)')
ax.set_title(f'CD99-CD81 Co-expression at Baseline\n'
             f'Spearman r={r:.3f}, p={p:.3f}')
sns.despine()
plt.tight_layout()
plt.savefig('cd99_cd81_integrated.pdf', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:



# Use raw counts for pseudobulk
treg_raw = treg.copy()
treg_raw.X = treg_raw.layers['counts']  # use raw counts layer

def pseudobulk(adata, groupby_cols):
    groups = adata.obs.groupby(groupby_cols)
    pb_counts = []
    pb_meta = []
    for name, idx in groups.indices.items():
        subset = adata[idx]
        counts = np.array(subset.X.sum(axis=0)).flatten()
        pb_counts.append(counts)
        meta = dict(zip(groupby_cols, name if isinstance(name, tuple) else [name]))
        meta['n_cells'] = len(idx)
        pb_meta.append(meta)
    pb = sc.AnnData(
        X=np.array(pb_counts),
        var=adata.var.copy(),
        obs=pd.DataFrame(pb_meta)
    )
    return pb

pb = pseudobulk(treg_raw, ['Patient', 'Treatment', 'Remission_status'])

# Ensure integer dtype
pb.X = np.round(pb.X).astype(int)

# Verify
print(pb.X.dtype)  # should be int
print(pb.X.min())  # should be >= 0

In [ ]:
pb.obs['Treatment'] = pb.obs['Treatment'].astype('category')
pb.obs['Remission_status'] = pb.obs['Remission_status'].astype('category')

# Filter lowly expressed genes
sc.pp.filter_genes(pb, min_cells=3)

# Create interaction term
pb.obs['interaction'] = (
    pb.obs['Treatment'].astype(str) + '_' + 
    pb.obs['Remission_status'].astype(str)
)
pb.obs['interaction'] = pb.obs['interaction'].astype('category')

In [ ]:
# Drop the separate factors and use only the combined interaction term
# which already encodes all the information

pb.obs['group'] = (
    pb.obs['Remission_status'].astype(str) + '_' + 
    pb.obs['Treatment'].astype(str)
)
pb.obs['group'] = pb.obs['group'].astype('category')

# Set reference level to Non_Remission_Pre
pb.obs['group'] = pd.Categorical(
    pb.obs['group'],
    categories=['Non_Remission_Pre', 'Non_Remission_Post', 
                'Remission_Pre', 'Remission_Post']
)

print(pb.obs['group'].value_counts())
print(f"N pseudobulk samples: {pb.n_obs}")

# Run DESeq2 with single combined factor
dds = DeseqDataSet(
    adata=pb,
    design_factors=['group'],
    ref_level=[['group', 'Non_Remission_Pre']],
    refit_cooks=True,
    n_cpus=8
)
dds.deseq2()

In [ ]:

# Extract interaction term results
# Genes where pre->post change differs by remission status
ds = DeseqStats(
    dds,
    contrast=['group', 'Remission_Post', 'Non_Remission_Post'],
    alpha=0.05,
    cooks_filter=True,
    independent_filter=True
)
ds.summary()
results = ds.results_df.dropna().sort_values('stat', ascending=False)

# Split into two gene sets
remission_resolved = results[
    (results['log2FoldChange'] > 0.5) & 
    (results['padj'] < 0.05)
].index.tolist()  # higher in remission post vs non-remission post

resistance_persistent = results[
    (results['log2FoldChange'] < -0.5) & 
    (results['padj'] < 0.05)
].index.tolist()  # higher in non-remission post vs remission post

print(f"Remission-resolved genes: {len(remission_resolved)}")
print(f"Resistance-persistent genes: {len(resistance_persistent)}")

In [ ]:

# All genes of interest
genes_of_interest = {
    'Remission-Resolved': remission_resolved,  # ['JUNB', 'RGS2', 'ZFP36L2', ...]
    'Resistance-Persistent': resistance_persistent  # ['VMP1', 'IFNG', 'SAMD9', ...]
}

palette = {
    ('Remission', 'Pre'):     {'color': '#378ADD', 'ls': '--', 'alpha': 0.8},
    ('Non_Remission', 'Pre'): {'color': '#D85A30', 'ls': '--', 'alpha': 0.8},
    ('Remission', 'Post'):    {'color': '#378ADD', 'ls': '-',  'alpha': 0.8},
    ('Non_Remission', 'Post'):{'color': '#D85A30', 'ls': '-',  'alpha': 0.8},
}

label_map = {
    ('Remission', 'Pre'):     'Remission Pre',
    ('Non_Remission', 'Pre'): 'Non-Remission Pre',
    ('Remission', 'Post'):    'Remission Post',
    ('Non_Remission', 'Post'):'Non-Remission Post',
}


In [ ]:

palette = {'Remission': '#378ADD', 'Non_Remission': '#D85A30'}

genes_of_interest = {
    'Remission-Resolved': remission_resolved,
    'Resistance-Persistent': resistance_persistent
}

for gene_set_name, gene_list in genes_of_interest.items():
    n_genes = len(gene_list)
    ncols = 3
    nrows = int(np.ceil(n_genes / ncols))

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(4.5 * ncols, 3.5 * nrows),
        sharey=False
    )
    axes = axes.flatten()

    for idx, gene in enumerate(gene_list):
        ax = axes[idx]

        if gene not in treg.var_names:
            ax.set_title(f'{gene} (not found)', fontsize=10)
            ax.axis('off')
            continue

        # Extract expression
        gene_expr = treg[:, gene].X
        if hasattr(gene_expr, 'toarray'):
            gene_expr = gene_expr.toarray().flatten()
        else:
            gene_expr = gene_expr.flatten()

        plot_df = treg.obs[['Patient', 'Treatment', 'Remission_status']].copy()
        plot_df['expression'] = gene_expr

        # Patient-level mean
        pt_df = plot_df.groupby(
            ['Patient', 'Remission_status', 'Treatment'],
            observed=True
        )['expression'].mean().reset_index()

        # Pivot pre/post
        pt_piv = pt_df.pivot_table(
            index=['Patient', 'Remission_status'],
            columns='Treatment',
            values='expression'
        ).reset_index().dropna()

        # Compute gene-level y range once
        y_min_gene = pt_piv[['Pre', 'Post']].min().min()
        y_max_gene = pt_piv[['Pre', 'Post']].max().max()
        y_range_gene = y_max_gene - y_min_gene

        x_positions = {
            'Remission':     {'Pre': 0,   'Post': 0.7},
            'Non_Remission': {'Pre': 1.2, 'Post': 1.9}
        }

        for status, color in palette.items():
            subset = pt_piv[pt_piv['Remission_status'] == status]
            if len(subset) == 0:
                continue

            x_pre  = x_positions[status]['Pre']
            x_post = x_positions[status]['Post']

            # Individual patient lines
            for _, row in subset.iterrows():
                ax.plot(
                    [x_pre, x_post], [row['Pre'], row['Post']],
                    color=color, alpha=0.35, lw=1.2
                )
                ax.scatter(
                    [x_pre, x_post], [row['Pre'], row['Post']],
                    color=color, s=30, zorder=3,
                    edgecolor='white', linewidth=0.6
                )

            # Group mean line
            ax.plot(
                [x_pre, x_post],
                [subset['Pre'].mean(), subset['Post'].mean()],
                color=color, lw=2.5, zorder=4
            )
            ax.scatter(
                [x_pre, x_post],
                [subset['Pre'].mean(), subset['Post'].mean()],
                color=color, s=80, zorder=5,
                edgecolor='white', linewidth=1.2
            )

            # Within-group Wilcoxon
            if len(subset) >= 4:
                diff = subset['Post'].values - subset['Pre'].values
                if not np.all(diff == 0):
                    try:
                        _, p = wilcoxon(subset['Pre'], subset['Post'])
                        ax.plot(
                            [x_pre, x_pre, x_post, x_post],
                            [y_max_gene + y_range_gene*0.05,
                             y_max_gene + y_range_gene*0.08,
                             y_max_gene + y_range_gene*0.08,
                             y_max_gene + y_range_gene*0.05],
                            'k-', lw=0.8
                        )
                        ax.text(
                            (x_pre + x_post) / 2,
                            y_max_gene + y_range_gene*0.10,
                            f'p={p:.3f}',
                            ha='center', fontsize=7
                        )
                    except ValueError:
                        pass

        # Axis formatting
        y_bottom = y_min_gene - y_range_gene * 0.28
        ax.set_xticks([0, 0.7, 1.2, 1.9])
        ax.set_xticklabels(['Pre', 'Post', 'Pre', 'Post'], fontsize=8)
        ax.set_ylabel('Normalized Expression', fontsize=8)
        ax.set_title(gene, fontsize=11, fontstyle='italic')
        ax.set_xlim(-0.3, 2.2)
        ax.set_ylim(
            y_min_gene - y_range_gene * 0.35,
            y_max_gene + y_range_gene * 0.20
        )
        ax.text(
            0.35, y_bottom, 'Remission',
            ha='center', color='#378ADD', fontsize=8, fontweight='bold'
        )
        ax.text(
            1.55, y_bottom, 'Non-Remission',
            ha='center', color='#D85A30', fontsize=8, fontweight='bold'
        )
        sns.despine(ax=ax)

    # Hide unused axes
    for i in range(n_genes, len(axes)):
        axes[i].axis('off')

    fig.suptitle(
        f'{gene_set_name} Genes\nPre vs Post Adalimumab',
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig(
        f'genes_prepost_{gene_set_name.replace(" ", "_").lower()}.pdf',
        dpi=300, bbox_inches='tight'
    )
    plt.show()

In [ ]:

palette = {'Remission': '#378ADD', 'Non_Remission': '#D85A30'}
x_positions = {
    'Remission':     {'Pre': 0,   'Post': 0.7},
    'Non_Remission': {'Pre': 1.2, 'Post': 1.9}
}

gene = 'CCL20'
gene_expr = treg[:, gene].X
if hasattr(gene_expr, 'toarray'):
    gene_expr = gene_expr.toarray().flatten()
else:
    gene_expr = gene_expr.flatten()

plot_df = treg.obs[['Patient', 'Treatment', 'Remission_status']].copy()
plot_df['expression'] = gene_expr

pt_df = plot_df.groupby(
    ['Patient', 'Remission_status', 'Treatment'], observed=True
)['expression'].mean().reset_index()

pt_piv = pt_df.pivot_table(
    index=['Patient', 'Remission_status'],
    columns='Treatment',
    values='expression'
).reset_index().dropna()

pt_piv['delta'] = pt_piv['Post'] - pt_piv['Pre']
rem = pt_piv[pt_piv['Remission_status'] == 'Remission']
non_rem = pt_piv[pt_piv['Remission_status'] == 'Non_Remission']

y_min_gene = pt_piv[['Pre', 'Post']].min().min()
y_max_gene = pt_piv[['Pre', 'Post']].max().max()
y_range_gene = y_max_gene - y_min_gene

fig, ax = plt.subplots(figsize=(4.8, 3.5))

for status, color in palette.items():
    subset = pt_piv[pt_piv['Remission_status'] == status]
    x_pre  = x_positions[status]['Pre']
    x_post = x_positions[status]['Post']

    for _, row in subset.iterrows():
        ax.plot(
            [x_pre, x_post], [row['Pre'], row['Post']],
            color=color, alpha=0.35, lw=1.2
        )
        ax.scatter(
            [x_pre, x_post], [row['Pre'], row['Post']],
            color=color, s=30, zorder=3,
            edgecolor='white', linewidth=0.6
        )

    ax.plot(
        [x_pre, x_post],
        [subset['Pre'].mean(), subset['Post'].mean()],
        color=color, lw=2.5, zorder=4
    )
    ax.scatter(
        [x_pre, x_post],
        [subset['Pre'].mean(), subset['Post'].mean()],
        color=color, s=80, zorder=5,
        edgecolor='white', linewidth=1.2
    )

_, p_delta = mannwhitneyu(
    rem['delta'].values, non_rem['delta'].values,
    alternative='two-sided'
)
_, p_baseline = mannwhitneyu(
    rem['Pre'].values, non_rem['Pre'].values,
    alternative='two-sided'
)

bracket_y   = y_max_gene + y_range_gene * 0.06
bracket_tip = y_max_gene + y_range_gene * 0.03
text_y      = y_max_gene + y_range_gene * 0.1

ax.plot(
    [0, 0, 1.9, 1.9],
    [bracket_tip, bracket_y, bracket_y, bracket_tip],
    'k-', lw=0.8
)
ax.text(
    0.95, text_y,
    f'Δ p={p_delta:.3f}' if p_delta >= 0.001 else 'Δ p<0.001',
    ha='center', fontsize=9
)

baseline_y    = y_min_gene - y_range_gene * 0.12
bracket_tip_b = y_min_gene - y_range_gene * 0.08

ax.plot(
    [0, 0, 1.2, 1.2],
    [bracket_tip_b, baseline_y, baseline_y, bracket_tip_b],
    'k-', lw=0.8
)
ax.text(
    0.6, baseline_y - y_range_gene * 0.08,
    f'p={p_baseline:.3f}' if p_baseline >= 0.001 else 'Baseline p<0.001',
    ha='center', fontsize=9
)

y_bottom = y_min_gene - y_range_gene * 0.38
ax.set_xticks([0, 0.7, 1.2, 1.9])
ax.set_xticklabels(['Pre', 'Post', 'Pre', 'Post'], fontsize=9)
ax.set_ylabel('CCL20 Expression', fontsize=10)
ax.set_xlim(-0.3, 2.2)
ax.set_ylim(
    y_min_gene - y_range_gene * 0.50,
    y_max_gene + y_range_gene * 0.22
)
ax.text(0.35, y_bottom, 'Remission',
        ha='center', color='#378ADD', fontsize=9, fontweight='bold')
ax.text(1.55, y_bottom, 'Non-Remission',
        ha='center', color='#D85A30', fontsize=9, fontweight='bold')
sns.despine(ax=ax)

print(f"CCL20: baseline p={p_baseline:.3f} | delta p={p_delta:.3f}")
plt.tight_layout()
plt.savefig('ccl20_prepost.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:

palette = {'Remission': '#378ADD', 'Non_Remission': '#D85A30'}
x_positions = {
    'Remission':     {'Pre': 0,   'Post': 0.7},
    'Non_Remission': {'Pre': 1.2, 'Post': 1.9}
}

gene = 'TNFRSF4'
gene_expr = treg[:, gene].X
if hasattr(gene_expr, 'toarray'):
    gene_expr = gene_expr.toarray().flatten()
else:
    gene_expr = gene_expr.flatten()

plot_df = treg.obs[['Patient', 'Treatment', 'Remission_status']].copy()
plot_df['expression'] = gene_expr

pt_df = plot_df.groupby(
    ['Patient', 'Remission_status', 'Treatment'], observed=True
)['expression'].mean().reset_index()

pt_piv = pt_df.pivot_table(
    index=['Patient', 'Remission_status'],
    columns='Treatment',
    values='expression'
).reset_index().dropna()

pt_piv['delta'] = pt_piv['Post'] - pt_piv['Pre']
rem = pt_piv[pt_piv['Remission_status'] == 'Remission']
non_rem = pt_piv[pt_piv['Remission_status'] == 'Non_Remission']

y_min_gene = pt_piv[['Pre', 'Post']].min().min()
y_max_gene = pt_piv[['Pre', 'Post']].max().max()
y_range_gene = y_max_gene - y_min_gene

fig, ax = plt.subplots(figsize=(4.8, 3.5))

for status, color in palette.items():
    subset = pt_piv[pt_piv['Remission_status'] == status]
    x_pre  = x_positions[status]['Pre']
    x_post = x_positions[status]['Post']

    for _, row in subset.iterrows():
        ax.plot(
            [x_pre, x_post], [row['Pre'], row['Post']],
            color=color, alpha=0.35, lw=1.2
        )
        ax.scatter(
            [x_pre, x_post], [row['Pre'], row['Post']],
            color=color, s=30, zorder=3,
            edgecolor='white', linewidth=0.6
        )

    ax.plot(
        [x_pre, x_post],
        [subset['Pre'].mean(), subset['Post'].mean()],
        color=color, lw=2.5, zorder=4
    )
    ax.scatter(
        [x_pre, x_post],
        [subset['Pre'].mean(), subset['Post'].mean()],
        color=color, s=80, zorder=5,
        edgecolor='white', linewidth=1.2
    )

_, p_delta = mannwhitneyu(
    rem['delta'].values, non_rem['delta'].values,
    alternative='two-sided'
)
_, p_baseline = mannwhitneyu(
    rem['Pre'].values, non_rem['Pre'].values,
    alternative='two-sided'
)

bracket_y   = y_max_gene + y_range_gene * 0.06
bracket_tip = y_max_gene + y_range_gene * 0.03
text_y      = y_max_gene + y_range_gene * 0.1

ax.plot(
    [0, 0, 1.9, 1.9],
    [bracket_tip, bracket_y, bracket_y, bracket_tip],
    'k-', lw=0.8
)
ax.text(
    0.95, text_y,
    f'Δ p={p_delta:.3f}' if p_delta >= 0.001 else 'Δ p<0.001',
    ha='center', fontsize=9
)

baseline_y    = y_min_gene - y_range_gene * 0.12
bracket_tip_b = y_min_gene - y_range_gene * 0.08

ax.plot(
    [0, 0, 1.2, 1.2],
    [bracket_tip_b, baseline_y, baseline_y, bracket_tip_b],
    'k-', lw=0.8
)
ax.text(
    0.6, baseline_y - y_range_gene * 0.08,
    f'p={p_baseline:.3f}' if p_baseline >= 0.001 else 'Baseline p<0.001',
    ha='center', fontsize=9
)

y_bottom = y_min_gene - y_range_gene * 0.38
ax.set_xticks([0, 0.7, 1.2, 1.9])
ax.set_xticklabels(['Pre', 'Post', 'Pre', 'Post'], fontsize=9)
ax.set_ylabel('TNFRSF4 Expression', fontsize=10)
ax.set_xlim(-0.3, 2.2)
ax.set_ylim(
    y_min_gene - y_range_gene * 0.50,
    y_max_gene + y_range_gene * 0.22
)
ax.text(0.35, y_bottom, 'Remission',
        ha='center', color='#378ADD', fontsize=9, fontweight='bold')
ax.text(1.55, y_bottom, 'Non-Remission',
        ha='center', color='#D85A30', fontsize=9, fontweight='bold')
sns.despine(ax=ax)

print(f"TNFRSF4: baseline p={p_baseline:.3f} | delta p={p_delta:.3f}")
plt.tight_layout()
plt.savefig('tnfrsf4_prepost.pdf', dpi=300, bbox_inches='tight')
plt.show()